# Phase 1b — Sample SQL Queries against the DuckDB Schema

This notebook documents the canonical exit-criterion query for Phase 1b plus five representative analytics queries against the 9-table DuckDB schema produced by the parser.

## What you need before running

1. **A sample DuckDB file.** Produce one with:
   ```bash
   uv run generator --board-profile=small --count=20 --seed=42 \
       --out=data/synthetic/ --start-date=2026-04-01 --end-date=2026-04-15
   uv run parser --input=data/synthetic/<run_dir>/ --db=data/db/sample.duckdb
   ```
   The cells below default to `data/db/sample.duckdb` relative to the repo root.

2. **Jupyter.** Not yet a project dependency; install ad-hoc with `uv pip install jupyter` or open this notebook in VS Code / Cursor (their notebook viewer brings its own kernel).

## Schema source-of-truth

All DDL lives in [`src/flying_probe_copilot/db/schema.py`](../src/flying_probe_copilot/db/schema.py). Nine tables, organised as 5 dimension + 1 metadata + 3 fact:

| Layer | Tables |
|---|---|
| Dimension | `boards`, `panels`, `operators`, `components`, `tests` |
| Metadata  | `runs` (one row per ingested generator run) |
| Fact      | `test_runs`, `measurements`, `failures` |

Surrogate PKs (`component_id`, `test_id`, `test_run_id`, `measurement_id`, `failure_id`) are assigned by the ingest layer using Python-side counters — DuckDB has no auto-increment in the Phase 1b DDL.

## Schema rationale

See [`docs/logs/DECISION_LOG.md`](../docs/logs/DECISION_LOG.md) under the 2026-06-14 Phase 1b entries:

- **boards-vs-panels split** — `boards` holds profile metadata (~3 rows ever); `panels` holds per-serial instances. Keeps "yield by board" semantically clean as per-profile aggregation.
- **Global `components` per (profile, refdes)** — avoids per-panel row explosion.
- **Limits persisted on `measurements`** — `limit_high` / `limit_low` / `limit_nominal` from `@LIM2` / `@LIM3` are nullable columns, irrecoverable from logs after the fact.
- **Denormalized `failures` table** — `panel_serial` + `board_profile_id` carried through so Pareto queries don't have to join.
- **`test_runs.operator_id` is nullable** — populated from `@BATCH.operator_id`, which is one value per log file. Per-panel operator recovery is deferred to Phase 2 (`results.json` sidecar or generator change). Treat per-operator yield in this schema as **per-run-operator** in v1.

## Setup

In [ ]:
from pathlib import Path
import duckdb

DB_PATH = Path("../data/db/sample.duckdb")
assert DB_PATH.exists(), f"No DuckDB file at {DB_PATH.resolve()} — generate one first (see README cell above)."

con = duckdb.connect(str(DB_PATH), read_only=True)

tables = [r[0] for r in con.execute("SHOW TABLES").fetchall()]
print(f"Tables ({len(tables)}):", ", ".join(tables))

con.execute(
    """
    SELECT run_id, board_profile_id, panel_count, failing_boards,
           fault_profile, fault_rate, ingested_at
    FROM runs
    ORDER BY ingested_at DESC
    """
).df()

## Query 1 — Yield by board over the last 7 days (canonical exit-criterion query)

The Phase 1b exit criterion in [`docs/ROADMAP.md`](../docs/ROADMAP.md) is:

> "Yield by board over last week" query returns correct result.

The window is anchored to `MAX(panels.scheduled_ts)` rather than `CURRENT_DATE` so the query is deterministic against any sample fixture (the sample data lives in April 2026, not today). In production use, swap the CTE for `CURRENT_TIMESTAMP - INTERVAL 7 DAY`.

`btest_status = 0` is `BTESTStatus.PASS` (see `flying_probe_copilot.generator.models.BTESTStatus`).

In [ ]:
con.execute(
    """
    WITH win AS (
        SELECT MAX(scheduled_ts) AS end_ts,
               MAX(scheduled_ts) - INTERVAL 7 DAY AS start_ts
        FROM panels
    )
    SELECT p.board_profile_id,
           COUNT(*)                                                   AS panels_tested,
           SUM(CASE WHEN tr.btest_status = 0 THEN 1 ELSE 0 END)       AS panels_passed,
           ROUND(100.0 * SUM(CASE WHEN tr.btest_status = 0 THEN 1 ELSE 0 END)
                       / COUNT(*), 2)                                 AS yield_pct
    FROM test_runs tr
    JOIN panels p ON p.panel_serial = tr.panel_serial
    CROSS JOIN win w
    WHERE p.scheduled_ts >= w.start_ts AND p.scheduled_ts <= w.end_ts
    GROUP BY p.board_profile_id
    ORDER BY p.board_profile_id
    """
).df()

## Query 2 — Failure Pareto by `record_type`

`failures` is denormalized on purpose (see DECISION_LOG 2026-06-14): `record_type` + `failure_category` + `panel_serial` + `board_profile_id` are carried so a Pareto needs no joins. `record_type` values come from `flying_probe_copilot.parser.ingest._record_type_str` — `TS` (shorts), `A-RES`/`A-CAP`/`A-IND`/`A-DIO`/… (analog), `D-T` (digital), `TJET`, `PF`.

In [ ]:
con.execute(
    """
    SELECT record_type,
           failure_category,
           COUNT(*)                                                AS failures,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)      AS pct_of_total
    FROM failures
    GROUP BY record_type, failure_category
    ORDER BY failures DESC
    """
).df()

## Query 3 — Per-shift yield

`shift` lives on `panels` (`A` / `B` / `C`). The small sample only covers one shift; on a medium/large run all three appear.

In [ ]:
con.execute(
    """
    SELECT p.shift,
           COUNT(*)                                                   AS panels_tested,
           SUM(CASE WHEN tr.btest_status = 0 THEN 1 ELSE 0 END)       AS panels_passed,
           ROUND(100.0 * SUM(CASE WHEN tr.btest_status = 0 THEN 1 ELSE 0 END)
                       / COUNT(*), 2)                                 AS yield_pct
    FROM test_runs tr
    JOIN panels p ON p.panel_serial = tr.panel_serial
    GROUP BY p.shift
    ORDER BY p.shift
    """
).df()

## Query 4 — Per-operator yield

**Caveat (DECISION_LOG 2026-06-14):** `test_runs.operator_id` in v1 is populated from `@BATCH.operator_id`, which is one value per per-board log file. Every panel in a single run therefore appears under the same operator. True per-panel operator attribution is deferred to Phase 2. Treat this query as **per-run-operator** until that lands.

In [ ]:
con.execute(
    """
    SELECT COALESCE(operator_id, '<unknown>')                       AS operator_id,
           COUNT(*)                                                 AS panels_tested,
           SUM(CASE WHEN btest_status = 0 THEN 1 ELSE 0 END)        AS panels_passed,
           ROUND(100.0 * SUM(CASE WHEN btest_status = 0 THEN 1 ELSE 0 END)
                       / COUNT(*), 2)                               AS yield_pct
    FROM test_runs
    GROUP BY operator_id
    ORDER BY panels_tested DESC, operator_id
    """
).df()

## Query 5 — Top-10 failing reference designators

`target_refdes` on `failures` is `NULL` for shorts records (which are panel-level rather than component-level), so the `WHERE` clause excludes them.

In [ ]:
con.execute(
    """
    SELECT f.target_refdes,
           c.component_family,
           COUNT(*) AS failures
    FROM failures f
    LEFT JOIN components c
      ON c.board_profile_id = f.board_profile_id
     AND c.refdes           = f.target_refdes
    WHERE f.target_refdes IS NOT NULL
    GROUP BY f.target_refdes, c.component_family
    ORDER BY failures DESC, f.target_refdes
    LIMIT 10
    """
).df()

## Query 6 — `btest_status` distribution

Maps numeric statuses on `test_runs.btest_status` to their human-readable category. The CASE values come straight from `flying_probe_copilot.generator.models.BTESTStatus`. Derivation order (categorical precedence: SHORTS → ANALOG → DIGITAL → PIN → TJET) is documented in DECISION_LOG 2026-06-13 (`@BTEST status derivation`).

In [ ]:
con.execute(
    """
    SELECT btest_status,
           CASE btest_status
               WHEN 0  THEN 'PASS'
               WHEN 1  THEN 'FAIL_UNCATEGORIZED'
               WHEN 2  THEN 'FAIL_PIN'
               WHEN 3  THEN 'FAIL_LEARN'
               WHEN 4  THEN 'FAIL_SHORTS'
               WHEN 6  THEN 'FAIL_ANALOG'
               WHEN 7  THEN 'FAIL_POWER'
               WHEN 8  THEN 'FAIL_DIGITAL'
               WHEN 9  THEN 'FAIL_FUNCTIONAL'
               WHEN 10 THEN 'FAIL_PRE_SHORTS'
               WHEN 14 THEN 'FAIL_TJET'
               WHEN 15 THEN 'FAIL_POLARITY'
               WHEN 16 THEN 'FAIL_CCHK'
               WHEN 17 THEN 'FAIL_ANALOG_CLUSTER'
               ELSE 'OTHER'
           END                                                     AS status_label,
           COUNT(*)                                                AS test_runs,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)      AS pct_of_total
    FROM test_runs
    GROUP BY btest_status
    ORDER BY test_runs DESC
    """
).df()

## Cleanup

In [ ]:
con.close()